In [1]:
import json
import pandas as pd

FILE = "Grice_by_Ben.txt"

rows = []
with open(FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))

df = pd.DataFrame(rows)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())

print("\nScore distribution:")
print(df["score"].value_counts(dropna=False))

Shape: (234192, 11)

Columns:
['text', 'related_text', 'reasoning', 'score', '_dialog_id', '_turn_index', '_unit_index', '_speaker', '_source_text', '_error', '_raw_output_first400']

First 5 rows:
                                                text  \
0                  am looking for a place to to stay   
1                         that has cheap price range   
2                    it should be in a type of hotel   
3  do you have a specific area you want to stay in ?   
4          no , i just need to make sure it 's cheap   

                        related_text                     reasoning score  \
0                                           Initial goal statement  Good   
1  am looking for a place to to stay   Specifies budget constraint  Good   
2                 a place to to stay  Clarifies accommodation type  Good   
3    it should be in a type of hotel        requests location info  Good   
4                  cheap price range     Reiterates budget concern  Good   

      _d

In [2]:
import json
import pandas as pd


OUTPUT = "grice_slim.csv"

score_map = {
    "Very poor": 1,
    "Poor": 2,
    "Weak": 3,
    "Neutral": 4,
    "Good": 5,
    "Very good": 6,
    "Excellent": 7,
}

rows = []
with open(FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        rows.append(obj)

df = pd.DataFrame(rows)

# Keep only rows with a valid score.
df_clean = df[df["score"].notna()].copy()

# Map score labels to numbers.
df_clean["score_num"] = df_clean["score"].map(score_map)

# Keep only the columns we need for analysis.
df_slim = df_clean[
    ["_dialog_id", "_turn_index", "_unit_index", "_speaker", "score", "score_num"]
].copy()

# Optional: sort for readability.
df_slim = df_slim.sort_values(
    by=["_dialog_id", "_turn_index", "_unit_index"]
).reset_index(drop=True)

# Save slim file.
df_slim.to_csv(OUTPUT, index=False, encoding="utf-8")

print("Slim shape:", df_slim.shape)
print("\nMissing numeric scores:", df_slim["score_num"].isna().sum())
print("\nFirst 10 rows:")
print(df_slim.head(10))
print("\nSaved file:", OUTPUT)

Slim shape: (234046, 6)

Missing numeric scores: 0

First 10 rows:
     _dialog_id  _turn_index  _unit_index  _speaker      score  score_num
0  MUL0001.json            0          0.0  customer       Good          5
1  MUL0001.json            1          0.0    expert       Good          5
2  MUL0001.json            1          1.0    expert  Very good          6
3  MUL0001.json            2          0.0  customer       Good          5
4  MUL0001.json            3          0.0    expert       Good          5
5  MUL0001.json            3          1.0    expert       Good          5
6  MUL0001.json            3          2.0    expert    Neutral          4
7  MUL0001.json            4          0.0  customer       Good          5
8  MUL0001.json            5          0.0    expert       Good          5
9  MUL0001.json            5          1.0    expert       Good          5

Saved file: grice_slim.csv


In [3]:
import pandas as pd

FILE = "grice_slim.csv"
OUTPUT = "grice_turn_level.csv"

df = pd.read_csv(FILE)

# Aggregate unit-level scores to one score per turn.
grice_turn = (
    df.groupby(["_dialog_id", "_turn_index", "_speaker"], as_index=False)
      .agg(
          grice_mean=("score_num", "mean"),
          grice_median=("score_num", "median"),
          grice_min=("score_num", "min"),
          grice_max=("score_num", "max"),
          grice_n_units=("score_num", "count"),
      )
      .sort_values(["_dialog_id", "_turn_index"])
      .reset_index(drop=True)
)

grice_turn.to_csv(OUTPUT, index=False, encoding="utf-8")

print("Turn-level shape:", grice_turn.shape)
print("\nFirst 10 rows:")
print(grice_turn.head(10))
print("\nSummary of number of units per turn:")
print(grice_turn["grice_n_units"].describe())
print("\nSaved file:", OUTPUT)

Turn-level shape: (142902, 8)

First 10 rows:
     _dialog_id  _turn_index  _speaker  grice_mean  grice_median  grice_min  \
0  MUL0001.json            0  customer    5.000000           5.0          5   
1  MUL0001.json            1    expert    5.500000           5.5          5   
2  MUL0001.json            2  customer    5.000000           5.0          5   
3  MUL0001.json            3    expert    4.666667           5.0          4   
4  MUL0001.json            4  customer    5.000000           5.0          5   
5  MUL0001.json            5    expert    5.000000           5.0          5   
6  MUL0001.json            6  customer    4.500000           4.5          4   
7  MUL0001.json            7    expert    5.666667           5.0          5   
8  MUL0001.json            8  customer    5.000000           5.0          5   
9  MUL0001.json            9    expert    5.000000           5.0          5   

   grice_max  grice_n_units  
0          5              1  
1          6            

In [4]:
import pandas as pd

FILE = "grice_turn_level.csv"
OUTPUT = "grice_dialog_level.csv"

df = pd.read_csv(FILE)

grice_dialog = (
    df.groupby(["_dialog_id"], as_index=False)
      .agg(
          grice_dialog_mean=("grice_mean", "mean"),
          grice_dialog_median=("grice_mean", "median"),
          grice_dialog_min=("grice_mean", "min"),
          grice_dialog_max=("grice_mean", "max"),
          grice_turn_count=("_turn_index", "count"),
          grice_total_units=("grice_n_units", "sum"),
      )
      .sort_values("_dialog_id")
      .reset_index(drop=True)
)

grice_dialog.to_csv(OUTPUT, index=False, encoding="utf-8")

print("Shape:", grice_dialog.shape)
print("\nColumns:")
print(grice_dialog.columns.tolist())

print("\nFirst 10 rows:")
print(grice_dialog.head(10))

print("\nSummary of dialogue mean scores:")
print(grice_dialog["grice_dialog_mean"].describe())

print("\nSaved file:", OUTPUT)

Shape: (10438, 7)

Columns:
['_dialog_id', 'grice_dialog_mean', 'grice_dialog_median', 'grice_dialog_min', 'grice_dialog_max', 'grice_turn_count', 'grice_total_units']

First 10 rows:
     _dialog_id  grice_dialog_mean  grice_dialog_median  grice_dialog_min  \
0  MUL0001.json           5.016667             5.000000          4.000000   
1  MUL0002.json           4.892857             5.000000          3.000000   
2  MUL0003.json           5.343750             5.500000          4.000000   
3  MUL0004.json           4.854167             5.000000          3.000000   
4  MUL0005.json           4.967593             5.000000          4.000000   
5  MUL0006.json           5.296875             5.000000          4.000000   
6  MUL0007.json           5.071429             5.000000          2.666667   
7  MUL0008.json           5.422619             5.166667          4.000000   
8  MUL0009.json           5.083333             5.000000          4.000000   
9  MUL0010.json           5.258333            

In [5]:
import pandas as pd

OVERLAP_FILE = "dialog_level_overlap.csv"
GRICE_FILE = "grice_dialog_level.csv"
OUTPUT = "merged_dialog_analysis.csv"

overlap = pd.read_csv(OVERLAP_FILE)
grice = pd.read_csv(GRICE_FILE)

# Rename for matching.
grice = grice.rename(columns={"_dialog_id": "dialog_id"})

merged = overlap.merge(grice, on="dialog_id", how="inner")

print("Merged shape:", merged.shape)
print("\nColumns:")
print(merged.columns.tolist())

print("\nRows per mode:")
print(merged["mode"].value_counts())

print("\nUnique dialog IDs:", merged["dialog_id"].nunique())

print("\nFirst 10 rows:")
print(merged.head(10))

merged.to_csv(OUTPUT, index=False, encoding="utf-8")
print("\nSaved file:", OUTPUT)

FileNotFoundError: [Errno 2] No such file or directory: 'dialog_level_overlap.csv'